In [1]:
%cd /kaggle/working

!pip -q install -U transformers sentencepiece spacy scikit-learn pandas tqdm nltk pytorch-lightning
!python -m spacy download en_core_web_sm -q

!rm -rf /kaggle/working/AlignScore
!git clone https://github.com/yuh-zha/AlignScore.git /kaggle/working/AlignScore

# Patch AdamW
MODEL_FILE = "/kaggle/working/AlignScore/src/alignscore/model.py"

with open(MODEL_FILE, "r", encoding="utf-8") as f:
    code = f.read()

code = code.replace(
    "from transformers import AdamW, get_linear_schedule_with_warmup, AutoConfig",
    "from torch.optim import AdamW\nfrom transformers import get_linear_schedule_with_warmup, AutoConfig"
)

with open(MODEL_FILE, "w", encoding="utf-8") as f:
    f.write(code)

# Patch load_from_checkpoint
INFER_FILE = "/kaggle/working/AlignScore/src/alignscore/inference.py"

with open(INFER_FILE, "r", encoding="utf-8") as f:
    code = f.read()

code = code.replace(
    "self.model = BERTAlignModel(model=model).load_from_checkpoint(checkpoint_path=ckpt_path, strict=False).to(self.device)",
    "self.model = BERTAlignModel.load_from_checkpoint(checkpoint_path=ckpt_path, model=model, strict=False).to(self.device)"
)

with open(INFER_FILE, "w", encoding="utf-8") as f:
    f.write(code)

import sys
sys.path.insert(0, "/kaggle/working/AlignScore/src")

from alignscore.alignscore import AlignScore

print("AlignScore ready ✅")

/kaggle/working
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 77.4 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Cloning into '/kaggle/working/AlignScore'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 202 (delta 36), reused 12 (delta 12), pack-reused 153 (from 1)
Receiving objects: 100% (202/202), 523.86 KiB | 6.02 MiB/s, done.
Resolving deltas: 100% (113/113), done.
AlignScore ready ✅


In [2]:
import os

CKPT_PATH = "/kaggle/working/AlignScore-base.ckpt"

if not os.path.exists(CKPT_PATH):
    !wget -q -O /kaggle/working/AlignScore-base.ckpt https://huggingface.co/yzha/AlignScore/resolve/main/AlignScore-base.ckpt

print("Checkpoint exists:", os.path.exists(CKPT_PATH))

Checkpoint exists: True


In [3]:
import os
import json
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

INPUT_JSON = "/kaggle/input/datasets/belalheshamfathy/zxzxzxz/qa_data_answer_focused_summarized.json"

OUTPUT_DIR = "/kaggle/working/alignscore_summary_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 8
DEVICE = 0 if torch.cuda.is_available() else -1
THRESHOLDS = [0.30, 0.40, 0.50, 0.60, 0.70]

# ============================================================
# LOAD JSON
# ============================================================

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

# 3000 records -> 6000 samples
MAX_RECORDS = 6000
data = data[:MAX_RECORDS]

print("Original records used:", len(data))

# ============================================================
# BUILD DATAFRAME
# ============================================================

rows = []

for item in data:

    knowledge = item.get("knowledge", "")
    question = item.get("question", "")

    right_answer = item.get("right_answer", "")
    hallucinated_answer = item.get("hallucinated_answer", "")

    rows.append({
        "knowledge": knowledge,
        "question": question,
        "answer": right_answer,
        "label": 1,
        "answer_type": "right_answer"
    })

    rows.append({
        "knowledge": knowledge,
        "question": question,
        "answer": hallucinated_answer,
        "label": 0,
        "answer_type": "hallucinated_answer"
    })

df = pd.DataFrame(rows)

df = df.dropna(
    subset=["knowledge", "question", "answer", "label"]
).reset_index(drop=True)

df["label"] = df["label"].astype(int)

print("\nEvaluation samples:", len(df))
print(df["label"].value_counts())

# ============================================================
# PREPARE INPUTS
# ============================================================

contexts = (
    "Knowledge:\n" + df["knowledge"].astype(str) +
    "\n\nQuestion:\n" + df["question"].astype(str)
).tolist()

claims = df["answer"].astype(str).tolist()

labels = df["label"].values

# ============================================================
# LOAD ALIGNSCORE
# ============================================================

scorer = AlignScore(
    model="roberta-base",
    batch_size=BATCH_SIZE,
    device=DEVICE,
    ckpt_path=CKPT_PATH,
    evaluation_mode="nli_sp"
)

# ============================================================
# SCORING
# ============================================================

scores = []

for i in tqdm(
    range(0, len(contexts), BATCH_SIZE),
    desc="AlignScore scoring"
):

    batch_scores = scorer.score(
        contexts=contexts[i:i+BATCH_SIZE],
        claims=claims[i:i+BATCH_SIZE]
    )

    scores.extend(batch_scores)

scores = np.array(scores, dtype=float)

df["alignscore"] = scores

# ============================================================
# THRESHOLD SEARCH
# ============================================================

results = []

for th in THRESHOLDS:

    preds = (scores >= th).astype(int)

    acc = accuracy_score(labels, preds)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary",
        pos_label=1,
        zero_division=0
    )

    cm = confusion_matrix(labels, preds)

    results.append({
        "method": "AlignScore + Summary",
        "threshold": th,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tn": cm[0, 0],
        "fp": cm[0, 1],
        "fn": cm[1, 0],
        "tp": cm[1, 1]
    })

results_df = pd.DataFrame(results).sort_values(
    "f1",
    ascending=False
)

BEST_THRESHOLD = float(results_df.iloc[0]["threshold"])

final_preds = (scores >= BEST_THRESHOLD).astype(int)

df["prediction"] = final_preds
df["correct"] = df["prediction"] == df["label"]

# ============================================================
# SAVE RESULTS
# ============================================================

df.to_csv(
    f"{OUTPUT_DIR}/alignscore_summary_predictions.csv",
    index=False
)

results_df.to_csv(
    f"{OUTPUT_DIR}/alignscore_summary_threshold_results.csv",
    index=False
)

# ============================================================
# FINAL REPORT
# ============================================================

precision, recall, f1, _ = precision_recall_fscore_support(
    labels,
    final_preds,
    average="binary",
    pos_label=1,
    zero_division=0
)

final_summary = pd.DataFrame([{
    "method": "AlignScore + Summary",
    "best_threshold": BEST_THRESHOLD,
    "accuracy": accuracy_score(labels, final_preds),
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "num_samples": len(df)
}])

final_summary.to_csv(
    f"{OUTPUT_DIR}/alignscore_summary_final_summary.csv",
    index=False
)

print("\nThreshold Results:")
display(results_df)

print("\nBest Threshold:", BEST_THRESHOLD)

print("\nClassification Report:")
print(
    classification_report(
        labels,
        final_preds,
        target_names=["Hallucinated", "Supported"],
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(labels, final_preds))

print("\nFinal Summary:")
display(final_summary)

print("\nSaved files:")
print(f"{OUTPUT_DIR}/alignscore_summary_predictions.csv")
print(f"{OUTPUT_DIR}/alignscore_summary_threshold_results.csv")
print(f"{OUTPUT_DIR}/alignscore_summary_final_summary.csv")

Original records used: 6000

Evaluation samples: 12000
label
1    6000
0    6000
Name: count, dtype: int64


Lightning automatically upgraded your loaded checkpoint from v1.7.7 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint AlignScore-base.ckpt`


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...


Threshold Results:


,method,threshold,accuracy,precision,recall,f1,tn,fp,fn,tp
0,AlignScore + Summary,0.3,0.792750,0.777708,0.819833,0.798215,4594,1406,1081,4919
1,AlignScore + Summary,0.4,0.776750,0.787532,0.758000,0.772484,4773,1227,1452,4548
2,AlignScore + Summary,0.5,0.751250,0.791868,0.681667,0.732647,4925,1075,1910,4090
3,AlignScore + Summary,0.6,0.716583,0.789614,0.590500,0.675694,5056,944,2457,3543
4,AlignScore + Summary,0.7,0.669167,0.779768,0.471500,0.587661,5201,799,3171,2829



Best Threshold: 0.3

Classification Report:
              precision    recall  f1-score   support

Hallucinated       0.81      0.77      0.79      6000
   Supported       0.78      0.82      0.80      6000

    accuracy                           0.79     12000
   macro avg       0.79      0.79      0.79     12000
weighted avg       0.79      0.79      0.79     12000


Confusion Matrix:
[[4594 1406]
 [1081 4919]]

Final Summary:


,method,best_threshold,accuracy,precision,recall,f1,num_samples
0,AlignScore + Summary,0.3,0.79275,0.777708,0.819833,0.798215,12000



Saved files:
/kaggle/working/alignscore_summary_outputs/alignscore_summary_predictions.csv
/kaggle/working/alignscore_summary_outputs/alignscore_summary_threshold_results.csv
/kaggle/working/alignscore_summary_outputs/alignscore_summary_final_summary.csv
